In [1]:
import os
import shutil
import albumentations as A
import numpy as np
import cv2 as cv
from PIL import Image
from albumentations.core.composition import OneOf
from albumentations.pytorch.transforms import ToTensorV2

In [2]:
def change_extention(file_name,new_extention):
    file,extention = file_name.rsplit('.', 1)
    return file+f'.{new_extention}'


In [3]:


def move_and_rename_images(source_dir, destination_dir, rename_pattern):
  os.makedirs(destination_dir+'\\images')
  os.makedirs(destination_dir+'\\labels')
  count = 0
  index=0
  classes=[]
  for train_root, _, files in os.walk(source_dir):
    labels_root=train_root.replace('train','labels')
    _,class_name=train_root.rsplit('\\',1)
    if(class_name=='train'):
      continue
    
    classes.append(class_name)
    
    for file in files:

      if file.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_src_path = os.path.join(train_root, file)
        lbl_src_path = os.path.join(labels_root, change_extention(file,'txt'))
         
        new_name = rename_pattern.format(count=count,extention='jpg', original_filename=file)
        img_dst_path = os.path.join(destination_dir+'\\images', new_name)
        lbl_dst_path = os.path.join(destination_dir+'\\labels', change_extention(new_name,'txt'))
        try:
          shutil.copy(lbl_src_path, lbl_dst_path)
          with open(lbl_dst_path, 'r+') as f:
            lines = f.readlines()
            f.seek(0)
            for line in lines:
              line = line.replace('0', str(index),1)
              f.write(line)
            f.truncate()
        except FileNotFoundError:
          continue
        shutil.copy(img_src_path, img_dst_path)
        count += 1
        with Image.open(img_dst_path) as img:
         rgb_img = img.convert('RGB')
         rgb_img.save(img_dst_path, 'JPEG')
    index+=1
  print(classes)

source_dir = "D:\\good\\archive(2)\\train"
destination_dir = "C:\\Users\\aa886\\OneDrive\\Desktop\\data"
rename_pattern = "image_{count}.{extention}"


In [4]:
shutil.rmtree(destination_dir)
move_and_rename_images(source_dir, destination_dir, rename_pattern)

c:\Users\aa886\AppData\Local\Programs\Python\Python39\lib\site-packages\PIL\Image.py:981: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


['apple', 'banana', 'beetroot', 'bell pepper', 'cabbage', 'capsicum', 'carrot', 'cauliflower', 'chilli pepper', 'corn', 'cucumber', 'eggplant', 'garlic', 'ginger', 'grapes', 'jalepeno', 'kiwi', 'lemon', 'lettuce', 'mango', 'onion', 'orange', 'paprika', 'pear', 'peas', 'pineapple', 'pomegranate', 'potato', 'raddish', 'soy beans', 'spinach', 'sweetcorn', 'sweetpotato', 'tomato', 'turnip', 'watermelon']


In [2]:
TARGET_HEIGHT=512

In [76]:
transform_pipeline = A.Compose([

    A.Affine(
        scale=(0.5, 1.5),
        translate_percent=(0.1, 0.1),
        rotate=0.0,
        shear=0.0,
        cval=(114, 114, 114),
        p=1.0,
    ),
    A.Blur(p=0.01),
    A.MedianBlur(p=0.01),
    A.ToGray(p=0.01),
    A.CLAHE(p=0.01),
    A.ColorJitter(
        contrast=0.0,
        saturation=0.7,
        hue=0.015,
        brightness=0.4,
    ),  
    A.HorizontalFlip(p=0.5),
    # letter box
  ],bbox_params = A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    ))

In [77]:
def get_annotations(annotation_file):
    with open(annotation_file, 'r') as f:
        lines = f.readlines()
    bboxes = []
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        bboxes.append([x_center, y_center, width, height])
    return class_id,bboxes


image = cv.imread("C:\\Users\\aa886\OneDrive\Desktop\\data\\images\\image_37.jpg")
annotations_file = "C:\\Users\\aa886\OneDrive\Desktop\\data\\labels\\image_37.txt"  
class_id,bboxes = get_annotations(annotations_file)


In [78]:
transformed = transform_pipeline(image=image, bboxes=bboxes,class_labels=[class_id]*len(bboxes))


transformed_image = transformed["image"]
transformed_bboxes = transformed["bboxes"]
print(transformed_bboxes)
for bbox in transformed_bboxes:
    x_center, y_center, width, height = bbox
    x_min = int((x_center - width/2) * image.shape[1])
    y_min = int((y_center - height/2) * image.shape[0])
    x_max = int((x_center + width/2) * image.shape[1])
    y_max = int((y_center + height/2) * image.shape[0])
    cv.rectangle(transformed_image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

cv.imshow("Transformed Image with Bounding Box", transformed_image)
cv.waitKey(0)
cv.destroyAllWindows()

[[0.5744740962982178, 0.637234091758728, 0.46680882573127747, 0.7255317568778992], [0.3665621280670166, 0.5491124391555786, 0.3515961766242981, 0.9017751216888428], [0.2225467562675476, 0.47870269417762756, 0.22910045087337494, 0.9392722249031067], [0.15798772871494293, 0.27926552295684814, 0.21519586443901062, 0.5585310459136963], [0.08048364520072937, 0.22535578906536102, 0.16096729040145874, 0.45071157813072205]]
